# Prompt-injection L1 model benchmark — Phase 0

Goal: pick the L1 model for `policy.default.yaml::guard.primary_model`.

Candidates:
- `meta-llama/Llama-Prompt-Guard-2-86M` (gated — needs `huggingface-cli login`)
- `protectai/deberta-v3-base-prompt-injection-v2` (~184M)
- `protectai/deberta-v3-small-prompt-injection-v2` (~44M)

Decision rule: prefer the smallest model whose **recall ≥ 80% on attacks** AND **FPR ≤ 5% on benign** AND **p95 CPU latency ≤ 30 ms**.

## 1. Install heavy deps (run once)

In [ ]:
%pip install -q transformers torch sentencepiece

## 2. Run the benchmark

All logic lives in `promptguard_benchmark.py` so that `eval/runner.py` can reuse `load_samples`.

In [ ]:
from notebooks.promptguard_benchmark import load_samples, run_model, print_summary

samples = load_samples()
print(f'{len(samples)} samples loaded')

models = [
    'protectai/deberta-v3-small-prompt-injection-v2',
    'protectai/deberta-v3-base-prompt-injection-v2',
    # 'meta-llama/Llama-Prompt-Guard-2-86M',  # uncomment after `huggingface-cli login`
]
results = [run_model(m, samples, device='cpu', threshold=0.5) for m in models]
print_summary(results)

## 3. Inspect misses / FPs of the leading model

In [ ]:
leader = next((r for r in results if not r.error), None)
if leader:
    print(f'=== {leader.model_id}: {len(leader.misses)} miss / {len(leader.fps)} FP ===')
    for m in leader.misses:
        print(f"  MISS [{m['category']}/{m['lang']}] score={m['score']:.2f} | {m['text'][:80]}")
    for m in leader.fps:
        print(f"  FP   [{m['category']}/{m['lang']}] score={m['score']:.2f} | {m['text'][:80]}")

## 4. Decision

Update `backend/config/policy.default.yaml::guard.primary_model` to the chosen model. Capture the recall / FPR / latency numbers in the demo slide.